# RAGLens — Corpus Builder

**Run this notebook once in Google Colab (free T4 GPU optional — CPU is fine).**

It will:
1. Download 50 RAG/LLM papers from arXiv (direct HTTP — no arxiv library needed)
2. Extract text from each PDF
3. Chunk, embed with BGE-small-en-v1.5, and build a FAISS index
4. Save `chunks.json` and `index.faiss`
5. Download both files — commit them to your repo

**After this notebook:** commit `corpus/processed/chunks.json` and `corpus/index.faiss` to your repo. The HuggingFace Space will load them directly at startup.

In [ ]:
# No arxiv library needed — we download PDFs directly via HTTP
!pip install -q pymupdf sentence-transformers faiss-cpu

In [ ]:
# 50 arXiv paper IDs covering RAG, retrieval, evaluation, and fine-tuning
PAPER_IDS = [
    # --- Foundational RAG ---
    "2005.11401",  # RAG: Retrieval-Augmented Generation for Knowledge-Intensive NLP (Lewis et al.)
    "2312.10997",  # Retrieval-Augmented Generation for Large Language Models: A Survey
    "2401.15884",  # RAG vs Fine-tuning: Pipelines, Tradeoffs, and a Case Study on Agriculture
    "2307.11019",  # Lost in the Middle: How LLMs Use Long Contexts
    "2310.01558",  # Self-RAG: Learning to Retrieve, Generate, and Critique
    "2302.00083",  # REPLUG: Retrieval-Augmented Language Model Pre-Training
    "2212.10560",  # Precise Zero-Shot Dense Retrieval without Relevance Labels (HyDE)
    "2209.01975",  # Atlas: Few-shot Learning with Retrieval Augmented Language Models
    # --- Dense Retrieval ---
    "2004.04906",  # Dense Passage Retrieval (DPR) (Karpukhin et al.)
    "2112.09118",  # BEIR: A Heterogeneous Benchmark for Zero-shot Retrieval
    "2010.01168",  # ColBERT: Efficient and Effective Passage Search via Contextualized Late Interaction
    "2205.04733",  # Improving Passage Retrieval with Zero-Shot Question Generation
    "2108.00573",  # In-Batch Negatives for Knowledge Distillation in Dense Retrieval
    "2305.14187",  # DRAGON: Diverse Augmentation for Dense Retrieval
    # --- BGE / Embeddings ---
    "2309.07597",  # C-Pack: Packaged Resources To Advance General Chinese Embedding (BGE paper)
    "2401.00368",  # Improving Text Embeddings with Large Language Models
    "2212.03533",  # MTEB: Massive Text Embedding Benchmark
    # --- Sparse Retrieval & Hybrid ---
    "2109.10086",  # SPLADE: Sparse Lexical and Expansion Model for First Stage Ranking
    "2104.07186",  # A Few Brief Notes on DeepImpact, COIL, and a Conceptual Framework for Learned Sparse Retrieval
    "2210.11934",  # Hybrid Retrieval and Reranking for Information-Seeking Conversations
    # --- Reranking ---
    "1901.04085",  # Passage Re-ranking with BERT (Nogueira & Cho)
    "2310.08484",  # RankGPT: LLMs as Zero-Shot Document Rerankers
    "2304.09542",  # FiD-Light: Efficient and Effective Retrieval-Augmented Text Generation
    # --- Fine-tuning / LoRA / QLoRA ---
    "2106.09685",  # LoRA: Low-Rank Adaptation of Large Language Models (Hu et al.)
    "2305.14314",  # QLoRA: Efficient Finetuning of Quantized LLMs (Dettmers et al.)
    "2210.11610",  # Scaling Instruction-Finetuned Language Models (FLAN)
    "2308.16823",  # Code Llama: Open Foundation Models for Code
    "2309.16609",  # Mistral 7B
    "2401.04088",  # Phi-2: The surprising power of small language models
    # --- Evaluation & RAGAS ---
    "2309.15217",  # RAGAS: Automated Evaluation of Retrieval Augmented Generation
    "2306.05685",  # Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena
    "2212.08073",  # Benchmarking Large Language Models in Complex Instructions Following
    "2307.03109",  # How Is ChatGPT's Behavior Changing over Time?
    "2303.08774",  # GPT-4 Technical Report
    # --- Chunking & Context ---
    "2406.16937",  # Evaluating Chunking Strategies for Retrieval
    "2401.18059",  # RAPTOR: Recursive Abstractive Processing for Tree-Organized Retrieval
    "2310.04408",  # Retrieve Anything to Augment Large Language Models
    # --- LLM Surveys ---
    "2303.18223",  # A Survey of Large Language Models (Zhao et al.)
    "2307.06435",  # LLM Survey: Challenges, Techniques, and Future Research Directions
    "2402.06196",  # A Survey on Hallucination in Large Language Models
    # --- Production RAG ---
    "2312.15166",  # ARES: An Automated Evaluation Framework for RAG Systems
    "2401.15203",  # CRUD-RAG: A Comprehensive Chinese Benchmark for RAG of LLMs
    "2404.16130",  # Evaluation of RAG Systems: A Framework for Systematic Assessment
    "2310.11511",  # HuggingGPT: Solving AI Tasks with ChatGPT and its Friends in HuggingFace
    # --- Attention & Architecture ---
    "2205.01068",  # FlashAttention: Fast and Memory-Efficient Exact Attention
    "2307.08621",  # Extending Context Window of Large Language Models via Positional Interpolation
    "2310.05736",  # Mistral: A 7B parameter language model with sliding window attention
    # --- Miscellaneous foundational ---
    "1810.04805",  # BERT: Pre-training of Deep Bidirectional Transformers (Devlin et al.)
    "2005.14165",  # Language Models are Few-Shot Learners (GPT-3)
    "2204.02311",  # PaLM: Scaling Language Modeling with Pathways
]

In [ ]:
import os, json, time, re
import requests
import fitz  # PyMuPDF

os.makedirs("corpus/raw", exist_ok=True)
os.makedirs("corpus/processed", exist_ok=True)

HEADERS = {"User-Agent": "RAGLens-corpus-builder/1.0 (research project; sridhar@example.com)"}

def download_paper(paper_id: str) -> str | None:
    """Download PDF directly from arxiv.org — no arxiv library needed."""
    out_path = f"corpus/raw/{paper_id}.pdf"
    if os.path.exists(out_path):
        return out_path
    # arXiv PDF URL is stable and public
    url = f"https://arxiv.org/pdf/{paper_id}.pdf"
    try:
        resp = requests.get(url, headers=HEADERS, timeout=60)
        resp.raise_for_status()
        # arXiv sometimes returns an HTML page instead of a PDF when rate-limited
        if resp.headers.get("Content-Type", "").startswith("text/html"):
            print(f"  [RATE-LIMITED] {paper_id} — retrying after 10s")
            time.sleep(10)
            resp = requests.get(url, headers=HEADERS, timeout=60)
            resp.raise_for_status()
        with open(out_path, "wb") as f:
            f.write(resp.content)
        time.sleep(2)  # polite delay — arxiv asks for 3s between requests
        return out_path
    except Exception as e:
        print(f"  [FAIL] {paper_id}: {e}")
        return None

print(f"Downloading {len(PAPER_IDS)} papers from arxiv.org...")
print("(~2 seconds per paper = ~2 minutes total)\n")

downloaded = []
for i, pid in enumerate(PAPER_IDS):
    path = download_paper(pid)
    if path:
        downloaded.append((pid, path))
    status = "OK" if path else "SKIP"
    print(f"  [{i+1:02d}/{len(PAPER_IDS)}] {pid} — {status}")

print(f"\nDownloaded {len(downloaded)}/{len(PAPER_IDS)} papers.")

In [ ]:
CHUNK_SIZE = 512   # words (approx tokens)
CHUNK_OVERLAP = 50

def extract_text(pdf_path: str) -> str:
    """Extract plain text from PDF, strip obvious noise."""
    doc = fitz.open(pdf_path)
    pages = []
    for page in doc:
        text = page.get_text("text")
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        # Drop bare page numbers and very short lines (headers/footers)
        lines = [l for l in lines if not re.match(r'^\d{1,4}$', l) and len(l) > 20]
        pages.append(" ".join(lines))
    return " ".join(pages)

def chunk_text(text: str, source: str) -> list[dict]:
    """Split text into overlapping word-based chunks."""
    words = text.split()
    step = CHUNK_SIZE - CHUNK_OVERLAP
    chunks = []
    for i, start in enumerate(range(0, len(words), step)):
        chunk_words = words[start : start + CHUNK_SIZE]
        if len(chunk_words) < 50:  # skip tiny trailing chunks
            continue
        chunks.append({
            "id": f"{source}_{i}",
            "text": " ".join(chunk_words),
            "source": source,
            "chunk_idx": i,
        })
    return chunks

all_chunks = []
for paper_id, path in downloaded:
    try:
        text = extract_text(path)
        chunks = chunk_text(text, paper_id)
        all_chunks.extend(chunks)
        print(f"  {paper_id}: {len(chunks)} chunks")
    except Exception as e:
        print(f"  {paper_id}: FAILED — {e}")

print(f"\nTotal chunks: {len(all_chunks)}")

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("Loading BGE-small-en-v1.5...")
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")

texts = [c["text"] for c in all_chunks]
print(f"Embedding {len(texts)} chunks...")
embeddings = embedder.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings.astype(np.float32))
print(f"FAISS index: {index.ntotal} vectors, dim={dim}")

In [ ]:
faiss.write_index(index, "corpus/index.faiss")
with open("corpus/processed/chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

print("Saved:")
print(f"  corpus/index.faiss          {os.path.getsize('corpus/index.faiss') / 1e6:.1f} MB")
print(f"  corpus/processed/chunks.json {os.path.getsize('corpus/processed/chunks.json') / 1e6:.1f} MB")

In [ ]:
# Sanity check — retrieve top 3 chunks for a test query
test_query = "What is QLoRA and when should you use it?"
q_vec = embedder.encode([test_query], normalize_embeddings=True)
D, I = index.search(q_vec.astype(np.float32), k=3)

print(f"Query: {test_query}\n")
for rank, idx in enumerate(I[0]):
    chunk = all_chunks[idx]
    print(f"Rank {rank+1} | {chunk['source']} chunk {chunk['chunk_idx']} | L2={D[0][rank]:.4f}")
    print(chunk['text'][:250] + "...\n")

# If the top results mention QLoRA/LoRA, retrieval is working correctly

In [ ]:
# Download both files to your computer
from google.colab import files
files.download("corpus/index.faiss")
files.download("corpus/processed/chunks.json")
print("\nNext steps:")
print("1. Place corpus/index.faiss in your repo at  corpus/index.faiss")
print("2. Place chunks.json in your repo at          corpus/processed/chunks.json")
print("3. git add corpus/ && git commit -m 'Add pre-built corpus index'")
print("4. git push — your HuggingFace Space will pick it up automatically")